In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from collections import Counter
from PIL import Image
from tqdm import tqdm
import zipfile
import shutil
from PIL import Image
import seaborn as sns
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torchvision import models
from tqdm import tqdm
import wandb
import torchvision
from torch.nn import CrossEntropyLoss
import torchmetrics
import torch

import torch.nn.functional as F


In [11]:
print(os.getcwd())
my_dir = os.getcwd()
zip_file_path = os.path.join(my_dir, "archive.zip")
extract_path = os.path.join(my_dir, "raw-img")

image_df = pd.read_csv('my_data.csv')
print(image_df)
dataset_path = extract_path

label_dict = {
    "cane": 0,
     "cavallo": 1 ,
     "elefante": 2 ,
     "farfalla": 3 ,
     "gallina": 4 ,
     "gatto": 5 ,
     "mucca": 6 ,
     "pecora": 7,
     "ragno" : 8,
     "scoiattolo": 9,
}

C:\Users\ahlad\OneDrive\Documents\NS_zadanie_2\NS_jupyter_2
      subdirectory_name                           image_name  \
0                  cane      OIF-e2bexWrojgtQnAPPcUfOWQ.jpeg   
1                  cane  OIP---A27bIBcUgX1qkbpZOPswHaFS.jpeg   
2                  cane  OIP---cByAiEbIxIAleGo9AqOQAAAA.jpeg   
3                  cane  OIP---ZIdwfUcJeVxnh47zppcQHaFj.jpeg   
4                  cane  OIP---ZRsOF7zsMqhW30WeF8-AHaFj.jpeg   
...                 ...                                  ...   
26123        scoiattolo  OIP-_U7JiIoYjbWPqmmmmdsvJwHaF5.jpeg   
26124        scoiattolo  OIP-_VBkNQd_MZI4xoemUb-FtAHaE7.jpeg   
26125        scoiattolo  OIP-_WyHKgREia-4VijlL6DNswHaFj.jpeg   
26126        scoiattolo  OIP-_xFGMN0UbYduHdiXQ1maZAHaIF.jpeg   
26127        scoiattolo  OIP-_XkUFCI2duAyKDD9utKQzgHaFc.jpeg   

       image_size_bytes  width  height image_type  label  
0                  9451    300     225       JPEG      0  
1                  9334    300     214       JPEG    

In [12]:
class ImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['image_path']
        label = self.dataframe.iloc[idx]['label']
        image = Image.open(img_path)

        if self.transform:
            image = self.transform(image)

        # Convert label to tensor
        label = torch.tensor(label, dtype=torch.long)

        return image, label

In [13]:
def save_model(model, filename, params):
    torch.save({
        'model_state_dict': model.state_dict(),
        'params': params
    }, filename)

In [14]:
def load_model(filename, device='cpu'):
    checkpoint = torch.load(filename, map_location=device)
    params = checkpoint['params']

    model = SimpleCNN(
        num_classes=params['num_classes'],
        input_size=(params['img_size'], params['img_size']),
        conv_layers_config=params['conv_layers'],
        conv_dropout_rates=params['conv_dropout_rates'],
        fc_dropout_rates=params['fc_dropout_rates']
    )

    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(params['device'])
    model.eval()
    return model, params

In [15]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes, input_size=(128, 128), conv_layers_config=None, conv_dropout_rates=None, fc_dropout_rates=None):

        super(SimpleCNN, self).__init__()

        if conv_layers_config is None:
            conv_layers_config = [32, 64, 128]
        if conv_dropout_rates is None:
            conv_dropout_rates = [0.3] * len(conv_layers_config)  #
        if fc_dropout_rates is None:
            fc_dropout_rates = [0.5, 0.5]

        layers = []
        in_channels = 3  # RGB input

        # Add Convolutional Layers with custom dropout rates
        for i, (out_channels, drop_rate) in enumerate(zip(conv_layers_config, conv_dropout_rates)):
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            layers.append(nn.Dropout(drop_rate))  # Add dropout after convolutional layer
            in_channels = out_channels

        self.features = nn.Sequential(*layers)

        # Dynamically calculate the flattened feature size after convolutions
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, *input_size)
            out = self.features(dummy_input)
            self.flattened_size = out.view(1, -1).shape[1]

        # Add Fully Connected Layers with custom dropout rates
        self.classifier = nn.Sequential(
            nn.Linear(self.flattened_size, 512),
            nn.ReLU(),
            nn.Dropout(fc_dropout_rates[0]),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [16]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, num_classes=10, weight=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.num_classes = num_classes
        self.weight = weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        inputs = F.softmax(inputs, dim=1)
        targets = targets.long()
        one_hot = torch.zeros_like(inputs).scatter_(1, targets.view(-1, 1), 1)
        p_t = (inputs * one_hot).sum(dim=1)
        loss = -self.alpha * (1 - p_t) ** self.gamma * torch.log(p_t + 1e-6)  # To prevent log(0)

        if self.weight is not None:
            loss = loss * self.weight[targets.view(-1)]

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


In [17]:
def get_criterion(params, class_weights=None):
    if params['loss_function'] == 'cross_entropy':
        # If no class weights are provided, use the default CrossEntropyLoss without weight
        if class_weights is not None:
            return nn.CrossEntropyLoss(weight=class_weights)
        else:
            return nn.CrossEntropyLoss()
    elif params['loss_function'] == 'focal':
        # If no class weights are provided, set the weight parameter to None
        if class_weights is not None:
            return FocalLoss(alpha=0.25, gamma=2.0, num_classes=params['num_classes'], weight=class_weights)
        else:
            return FocalLoss(alpha=0.25, gamma=2.0, num_classes=params['num_classes'], weight=None)
    else:
        raise ValueError(f"Unsupported loss type: {params['loss_function']}")


In [18]:
def evaluate_model(model, val_loader, params, val_class_weights=None, epoch=None):
    model.eval()

    criterion = get_criterion(params, class_weights=class_weights)

    stats_val = {"loss_val": 0.0, "acc_val": 0.0}
    num_batches = len(val_loader)

    #print(f"\nEvaluating Epoch {epoch if epoch is not None else '-'}")

    with torch.no_grad():
        with tqdm(val_loader, desc=f"Val Epoch {epoch}", leave=False, dynamic_ncols=True) as progress:
            for x, y in progress:
                x, y = x.to(params['device']), y.to(params['device'])
                y_hat = model(x)
                loss = criterion(y_hat, y)

                stats_val["loss_val"] += loss.item() / num_batches
                stats_val["acc_val"] += compute_accuracy(y_hat, y) / num_batches

                progress.set_postfix({k: f"{v:.3f}" for k, v in stats_val.items()})

    print(f"Val Epoch {epoch if epoch is not None else '-'} completed | Loss: {stats_val['loss_val']:.4f}, Acc: {stats_val['acc_val']*100:.5f}%")

    return stats_val["loss_val"], stats_val["acc_val"]

In [19]:
from tqdm import tqdm

def compute_accuracy(y_hat, y):
    _, predicted = torch.max(y_hat, 1)
    correct = (predicted == y).sum().item()
    return correct / y.size(0)

def train_model(model, train_loader, val_loader, params, class_weights=None):
    model.to(params['device'])

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = get_criterion(params, class_weights=class_weights)

    # Training Loop
    for epoch in range(params['num_epochs']):
        model.train()  # Set the model to training mode
        stats_train = {"loss_train": 0.0, "acc_train": 0.0}
        num_batches = len(train_loader)

        #print(f"\nTraining Epoch {epoch+1}/{params['num_epochs']}")

        with tqdm(train_loader, desc=f"Train Epoch {epoch+1}", leave=False, dynamic_ncols=True) as progress:
            for x, y in progress:
                x, y = x.to(params['device']), y.to(params['device'])

                optimizer.zero_grad()
                y_hat = model(x)
                loss = criterion(y_hat, y)
                loss.backward()
                optimizer.step()


                stats_train["loss_train"] += loss.item() / num_batches
                stats_train["acc_train"] += compute_accuracy(y_hat, y) / num_batches

                progress.set_postfix({k: f"{v:.3f}" for k, v in stats_train.items()})



        #print(f"Train Epoch {epoch+1} completed | Loss: {stats_train['loss_train']:.4f}, Acc: {stats_train['acc_train']*100:.5f}%")

        inv_val = get_inverted_class_distribution(val_df)
        val_class_weights = torch.tensor(inv_val, dtype=torch.float32).to(params['device'])
        val_loss, val_acc = evaluate_model(model, val_loader, params, val_class_weights=val_class_weights, epoch=epoch+1)



        wandb.log({
            'train_loss': stats_train["loss_train"],
            'train_acc': stats_train["acc_train"],
            'val_loss': val_loss,
            'val_acc': val_acc
        })



<h3>Rozdelenie dát</h3>
Dáta rozdelujeme 80/10/10
Vytvárame výhy pre criterion na základe početností



In [20]:
params = {
    'batch_size': 32,
    'learning_rate': 1e-3,
    'num_epochs': 5,
    'img_size': 64,
    'train_val_split': 0.8,
    'val_split': 0.1,
    'num_classes': 10,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'conv_layers': [32, 64, 128],
    'conv_dropout_rates': [0.3, 0.4, 0.5],
    'fc_dropout_rates': [0.4, 0.5],
    'loss_function': 'focal',
    'focal_gamma': 2.0

}

In [21]:
image_df['image_path'] = image_df.apply(lambda row: os.path.join(dataset_path, row['subdirectory_name'], row['image_name']), axis=1)

# Shuffle the dataset (in-place)
image_df = image_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Manually splitting the data
train_size = int(params["train_val_split"] * len(image_df))  # 80% for training
val_size = int(params["val_split"] * len(image_df))    # 10% for validation
test_size = len(image_df) - train_size - val_size  # Remaining 10% for testing

train_df = image_df[:train_size]
val_df = image_df[train_size:train_size + val_size]
test_df = image_df[train_size + val_size:]

print(f"Training data size: {len(train_df)}")
print(f"Validation data size: {len(val_df)}")
print(f"Test data size: {len(test_df)}")

print(train_df.head())

Training data size: 20902
Validation data size: 2612
Test data size: 2614
  subdirectory_name                           image_name  image_size_bytes  \
0             ragno  OIP-iJAyy_puzMN2v-189vEsUwHaE8.jpeg             19351   
1           cavallo  OIP-PfzazZlyqJs4e-CIlPrQOAHaFj.jpeg             14429   
2           gallina                             607.jpeg             22977   
3           cavallo  OIP-AHSrAWtCAg6eLf1ulS9sZwHaE7.jpeg             13718   
4             ragno  OIP-Yd9Y4FzpKizjhvunnkQx9QHaIH.jpeg             10923   

   width  height image_type  label  \
0    300     200       JPEG      8   
1    300     225       JPEG      1   
2    300     225       JPEG      4   
3    300     200       JPEG      1   
4    274     300       JPEG      8   

                                          image_path  
0  C:\Users\ahlad\OneDrive\Documents\NS_zadanie_2...  
1  C:\Users\ahlad\OneDrive\Documents\NS_zadanie_2...  
2  C:\Users\ahlad\OneDrive\Documents\NS_zadanie_2...  
3  C:\Us

In [22]:
def print_class_distribution(df, split_name):
    print(f"\nClass distribution in {split_name} set:")
    print(df['subdirectory_name'].value_counts())

# Print class distributions for each split
print_class_distribution(train_df, "Train")
print_class_distribution(val_df, "Validation")
print_class_distribution(test_df, "Test")


Class distribution in Train set:
subdirectory_name
cane          3906
ragno         3793
gallina       2456
cavallo       2089
farfalla      1678
mucca         1505
scoiattolo    1476
pecora        1457
gatto         1360
elefante      1182
Name: count, dtype: int64

Class distribution in Validation set:
subdirectory_name
ragno         507
cane          475
gallina       322
cavallo       263
farfalla      202
scoiattolo    201
pecora        193
mucca         178
gatto         162
elefante      109
Name: count, dtype: int64

Class distribution in Test set:
subdirectory_name
ragno         520
cane          482
gallina       320
cavallo       271
farfalla      193
scoiattolo    185
mucca         183
pecora        170
elefante      145
gatto         145
Name: count, dtype: int64


In [23]:
def print_class_distribution_percent(df, split_name):
    print(f"\n{split_name} Set Class Distribution:")
    total = len(df)

    # Count occurrences of each class
    distribution = df['subdirectory_name'].value_counts().sort_index()

    # Convert to DataFrame
    distribution_df = distribution.reset_index()
    distribution_df.columns = ['class', 'count']

    # Add percentage column
    distribution_df['percentage'] = 100 * distribution_df['count'] / total

    # Print nicely formatted table
    print(distribution_df.to_string(index=False, formatters={'percentage': '{:.2f}%'.format}))

print_class_distribution_percent(train_df, "Train")
print_class_distribution_percent(val_df, "Validation")
print_class_distribution_percent(test_df, "Test")


Train Set Class Distribution:
     class  count percentage
      cane   3906     18.69%
   cavallo   2089      9.99%
  elefante   1182      5.65%
  farfalla   1678      8.03%
   gallina   2456     11.75%
     gatto   1360      6.51%
     mucca   1505      7.20%
    pecora   1457      6.97%
     ragno   3793     18.15%
scoiattolo   1476      7.06%

Validation Set Class Distribution:
     class  count percentage
      cane    475     18.19%
   cavallo    263     10.07%
  elefante    109      4.17%
  farfalla    202      7.73%
   gallina    322     12.33%
     gatto    162      6.20%
     mucca    178      6.81%
    pecora    193      7.39%
     ragno    507     19.41%
scoiattolo    201      7.70%

Test Set Class Distribution:
     class  count percentage
      cane    482     18.44%
   cavallo    271     10.37%
  elefante    145      5.55%
  farfalla    193      7.38%
   gallina    320     12.24%
     gatto    145      5.55%
     mucca    183      7.00%
    pecora    170      6.50%
    

In [24]:
def get_inverted_class_distribution(df):
    total = len(df)

    distribution = df['subdirectory_name'].value_counts().sort_index()

    percentages = distribution / total

    inverted_percentages = 3.0 - (3 * percentages)

    return inverted_percentages.values

inv_train = get_inverted_class_distribution(train_df)
inv_val = get_inverted_class_distribution(val_df)
inv_test = get_inverted_class_distribution(test_df)

print("Inverted Train:", inv_train)
print("Inverted Val:", inv_val)
print("Inverted Test:", inv_test)

Inverted Train: [2.43938379 2.70017223 2.83035116 2.7591618  2.64749785 2.80480337
 2.78399196 2.79088126 2.45560233 2.78815424]
Inverted Val: [2.45444104 2.69793262 2.87480858 2.76799387 2.63016845 2.81393568
 2.79555896 2.77833078 2.4176876  2.76914242]
Inverted Test: [2.44682479 2.6889824  2.83358837 2.77850038 2.63274675 2.83358837
 2.78997705 2.80489671 2.40321347 2.78768171]


<h1>Testing</h2>


In [25]:
params = {
    'batch_size': 32,
    'learning_rate': 0.001,
    'num_epochs': 5,
    'img_size': 128,
    'train_val_split': 0.8,
    'val_split': 0.1,
    'num_classes': 10,  # Number of classes in the dataset
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'conv_layers': [32, 64, 128, 128 , 256 , 256],  # Number of channels per convolutional layer
    'conv_dropout_rates': [0.0, 0.0, 0.0 ,0.0 , 0.0 , 0.0],  # Custom dropout for conv layers
    'fc_dropout_rates': [0.4, 0.5],  # Custom dropout for fully connected layers
    'loss_function': 'cross_entropy',  # Use Focal Loss
    'focal_gamma': 2.0
}

transform = transforms.Compose([
    transforms.Resize((params['img_size'], params['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageDataset(train_df, transform=transform)
val_dataset = ImageDataset(val_df, transform=transform)
test_dataset = ImageDataset(test_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=params['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False)


device = torch.device(params['device'])

if device.type == 'cuda':
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Training on CPU")

inv_train = get_inverted_class_distribution(train_df)
class_weights = torch.tensor(inv_train, dtype=torch.float32).to(params['device'])

inv_test = get_inverted_class_distribution(test_df)
test_class_weights = torch.tensor(inv_test, dtype=torch.float32).to(params['device'])


Training on GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [27]:
params = {
    'batch_size': 32,
    'learning_rate': 0.001,
    'num_epochs': 100,
    'img_size': 128,
    'train_val_split': 0.8,
    'val_split': 0.1,
    'num_classes': 10,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'conv_layers': [32,64,128,128,256,256],
    'conv_dropout_rates': [0.3, 0.3, 0.3, 0.3, 0.3, 0.3],
    'fc_dropout_rates': [0.4, 0.5],
    'loss_function': 'cross_entropy',
    'focal_gamma': 2.0
}

model = SimpleCNN(num_classes=params['num_classes'],input_size=(params['img_size'], params['img_size']),conv_layers_config=params['conv_layers'], conv_dropout_rates=params['conv_dropout_rates'], fc_dropout_rates=params['fc_dropout_rates'] )

run = wandb.init(project="NS_global_test", name="Best_No_weights", config=params)
train_model(model, train_loader, val_loader, params )
run = wandb.finish()

Val Epoch 1 completed | Loss: 1.6054, Acc: 45.52591%


Val Epoch 2 completed | Loss: 1.2644, Acc: 57.50000%


Val Epoch 3 completed | Loss: 1.2816, Acc: 55.88415%


Val Epoch 4 completed | Loss: 1.2244, Acc: 60.70122%


Val Epoch 5 completed | Loss: 0.9934, Acc: 66.04421%


...

Val Epoch 97 completed | Loss: 0.7121, Acc: 84.10061%


Val Epoch 98 completed | Loss: 0.7396, Acc: 83.91006%


Val Epoch 99 completed | Loss: 0.7331, Acc: 83.60518%


Val Epoch 100 completed | Loss: 0.7828, Acc: 83.39177%


train_acc,▁▃▄▆▆▆▆▇▇▇▇▇▇▇██████████████████████████
train_loss,█▆▆▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▄▅▅▆▇▇▇▇▇▇▇▇▇█████████▇███████████████
val_loss,█▄▃▂▃▂▂▂▂▂▁▁▂▁▂▁▂▂▂▁▂▂▂▂▁▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂
train_acc,0.95413
train_loss,0.13836
val_acc,0.83392
val_loss,0.78283


In [28]:
def summarize_mismatches(model, test_loader, label_dict, device, csv_filename="mismatch_summary.csv"):
    model.eval()
    mismatch_counter = Counter()

    inv_label_dict = {v: k for k, v in label_dict.items()}

    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc="Evaluating"):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for true, pred in zip(targets, preds):
                if true.item() != pred.item():
                    true_label = inv_label_dict[true.item()]
                    pred_label = inv_label_dict[pred.item()]
                    mismatch_counter[(true_label, pred_label)] += 1

    # Convert to DataFrame
    mismatch_data = [
        {"Expected": k[0], "Predicted": k[1], "Count": v}
        for k, v in mismatch_counter.items()
    ]
    mismatch_df = pd.DataFrame(mismatch_data)

    # Optional: sort by most frequent errors
    mismatch_df = mismatch_df.sort_values(by="Count", ascending=False).reset_index(drop=True)

    # Save to CSV
    mismatch_df.to_csv(csv_filename, index=False)
    print(f"Saved mismatch summary to {csv_filename}")

    return mismatch_df

In [29]:
summary_df = summarize_mismatches(model, test_loader, label_dict, device)
print(summary_df.head())

Evaluating: 100%|██████████| 82/82 [00:08<00:00,  9.54it/s]


Saved mismatch summary to mismatch_summary.csv
   Expected Predicted  Count
0     mucca    pecora     33
1  farfalla     ragno     28
2     gatto      cane     28
3   cavallo      cane     19
4   gallina      cane     18


In [30]:
summary_df.to_csv("no_weights.csv", index=False)


In [31]:
# Save
save_model(model, 'no_weights.pth', params)


In [56]:
loaded_model, loaded_params = load_model('my_model.pth')

In [57]:
summary_df = summarize_mismatches(loaded_model, test_loader, label_dict, device)
print(summary_df.head())

Evaluating: 100%|██████████| 82/82 [00:11<00:00,  7.12it/s]


Saved mismatch summary to mismatch_summary.csv
   Expected Predicted  Count
0     gatto      cane     26
1     mucca    pecora     21
2  farfalla     ragno     20
3   cavallo     mucca     20
4      cane    pecora     19
